# LC #853 — Car Fleet
**Category:** Stack / Sorting | **Difficulty:** Medium | **Day 3**

---

<div style="padding:15px;border-left:8px solid #f7971e;background:#fff8e1;border-radius:4px;">
<strong>The Problem:</strong> <code>n</code> cars head to <code>target</code> miles away.
Given <code>position[]</code> and <code>speed[]</code>. Cars that catch up to a car ahead
form a fleet (the faster car slows to the slower car's speed). Return the number of fleets.
</div>

**Example:**
```
target=12, position=[10,8,0,5,3], speed=[2,4,1,1,3] → 3
```
*Car at 10 arrives at t=1. Car at 8 arrives at t=1 too — same fleet. Others form 2 separate fleets.*

**Core Insight:** Sort by position descending. Calculate time to target for each car. If a car's arrival time is ≤ the car ahead's, they join the same fleet (the slower car's time dominates).

## Mental Models

**1. Arrival Time is the Key**
`time = (target - position) / speed` — how long to reach target at constant speed.
If car B (behind) arrives before or at the same time as car A (ahead): B catches up → same fleet. B's actual arrival time is then A's (it slows to A's speed).

**2. Sort Descending = Process Closest First**
Process cars from closest to target to furthest. A car can only join the fleet ahead of it (closer to target). It cannot "jump over" a car.

**3. Stack = Fleet Representatives**
The stack holds arrival times of distinct fleets. If the current car arrives after the fleet ahead (top of stack), it forms a new fleet. Otherwise it joins the existing one.

In [ ]:
# Simulation approach — sort and check
# O(n log n) — same as optimal but more verbose

def carFleet_verbose(target: int, position: list[int], speed: list[int]) -> int:
    cars = sorted(zip(position, speed), reverse=True)  # closest to target first
    fleets = []

    for pos, spd in cars:
        time = (target - pos) / spd
        # If no fleet ahead OR this car arrives after the current lead fleet
        if not fleets or time > fleets[-1]:
            fleets.append(time)   # new fleet
        # else: joins the fleet ahead (its time is absorbed — not added)

    return len(fleets)

# Test
print(carFleet_verbose(12, [10,8,0,5,3], [2,4,1,1,3]))   # 3
print(carFleet_verbose(10, [3], [3]))                       # 1
print(carFleet_verbose(100, [0,2,4], [4,2,1]))             # 1

## Trace: `target=12, position=[10,8,0,5,3], speed=[2,4,1,1,3]`

After sorting by position descending: `[(10,2), (8,4), (5,1), (3,3), (0,1)]`

| car (pos,spd) | time=(12-pos)/spd | stack (fleet times) | Action |
|---------------|-------------------|---------------------|--------|
| (10, 2) | (12-10)/2 = **1.0** | [1.0] | new fleet |
| (8, 4) | (12-8)/4 = **1.0** | [1.0] | 1.0 ≤ 1.0 → joins fleet |
| (5, 1) | (12-5)/1 = **7.0** | [1.0, 7.0] | 7.0 > 1.0 → new fleet |
| (3, 3) | (12-3)/3 = **3.0** | [1.0, 7.0] | 3.0 ≤ 7.0 → joins fleet |
| (0, 1) | (12-0)/1 = **12.0** | [1.0, 7.0, 12.0] | 12.0 > 7.0 → new fleet |

Result: **3 fleets**. Stack length = number of fleets.

In [ ]:
# Optimal — O(n log n) time (sort dominates), O(n) space

def carFleet(target: int, position: list[int], speed: list[int]) -> int:
    pairs = sorted(zip(position, speed), reverse=True)  # closest to target first
    stack = []   # arrival times of distinct fleets

    for pos, spd in pairs:
        time = (target - pos) / spd
        # New fleet if: stack empty OR arrives AFTER the fleet ahead
        if not stack or time > stack[-1]:
            stack.append(time)
        # else: joins the fleet ahead (slower car determines fleet arrival)

    return len(stack)

# Test
print(carFleet(12, [10, 8, 0, 5, 3], [2, 4, 1, 1, 3]))   # 3
print(carFleet(10, [3], [3]))                               # 1
print(carFleet(100, [0, 2, 4], [4, 2, 1]))                # 1  — all join same fleet
print(carFleet(10, [6, 8], [3, 2]))                        # 2  — separate fleets

## Complexity Analysis

| Step | Time | Notes |
|------|------|-------|
| Sort | O(n log n) | Dominates |
| Stack scan | O(n) | Single pass |
| **Total** | **O(n log n)** | **Sort bound** |

**Space:** O(n) — the stack holds at most n fleet arrival times.

**Why sorting by position descending matters:** A car can only catch up to cars closer to the target. Processing closest-first ensures we evaluate each car's ability to catch the car immediately ahead.

## Interview Q&A

**Q: Why sort by position descending?**
A: Cars can only catch up to cars ahead (closer to target). Processing closest-to-target first means when we evaluate a car, we can immediately compare it to the fleet it would join.

**Q: Why is arrival time the deciding factor?**
A: If car B (behind) arrives before or at the same time as car A (ahead): B physically catches A before the target, joins A's fleet, and slows to A's speed. B's arrival time becomes A's time.

**Q: What if two cars start at the same position?**
A: The problem states all positions are unique.

**Q: When does time > stack[-1] form a new fleet?**
A: When a car arrives *after* the fleet ahead — it can never catch up. It forms its own fleet with all cars behind it that join it.

## The Citi Angle

**The "merging groups" pattern:** Batch jobs or service requests that start at different times but converge on the same resource.

```python
# How many distinct "batches" will reach the database given staggered starts?
# Jobs that arrive within the same second merge into one batch

def count_db_batches(submissions: list[tuple]) -> int:
    """
    submissions: list of (ready_time, processing_rate) pairs.
    Jobs that "catch up" to the job ahead merge into one batch.
    """
    # Sort by ready_time descending (most ready first)
    jobs = sorted(submissions, reverse=True)
    batch_arrivals = []

    db_capacity = 1.0   # 1 unit of work per second
    for ready_time, rate in jobs:
        arrival = ready_time + (1.0 / rate)   # time to complete 1 unit
        if not batch_arrivals or arrival > batch_arrivals[-1]:
            batch_arrivals.append(arrival)

    return len(batch_arrivals)

# 4 jobs at different ready times with different processing rates
submissions = [(0, 2), (1, 1), (2, 3), (5, 1)]
print(f"Distinct DB batches: {count_db_batches(submissions)}")
```

**Interview tie-in:** "Car Fleet maps to any resource contention problem — multiple requestors converging on a shared bottleneck. The stack tracks distinct arrival groups, which equals distinct load spikes on the resource."

## Summary

| | Value |
|--|--|
| **Pattern** | Sort + Stack |
| **Time** | O(n log n) |
| **Space** | O(n) |
| **Key step** | Sort descending by position; compare arrival times |
| **Say in interview** | "Sort closest-to-target first. For each car, compute arrival time. If it arrives after the fleet ahead, it's a new fleet. Stack length = fleet count." |